# NB1: Baseline

Relevant to chapter 9.3.3 from the Understanding Deep Learning Book.

**Theory Claim**: Dropout removes kinks in regions far from the training data that don't affect the loss. With more training data, the training points cover the input space more desnely, so there are fewere unconstrained regions where those kinks can form.
*Taken from Understanding Deep Learning Page...*

**Our question**: Does the improvement in performance of neural networks hold when using dropout regularisation with increasing training dataset sizes?

## Experiment Information

**Our Baseline**: A model trained on a dataset with minimally available training images where dropout is increasingly applied.
Our model:
-  Uses 10% percent of training data.
- Every model has 256 hidden units and 2 hidden layers so there are more weights than data (roughly 45 per image).
- Optimiser Adam is used so that the model fits quickly and within 20 epochs to ensure that the model can recognise the patterns within the results. We trialed with epochs less than 20 that gave us minimally distingushable results and with more than 20 the results became too extreme.

Therefore, by using these as standardised parameters we have guaranteed that the model has been trained acceptably which is vital to the baseline as we want to show that dropout improves the models performance.


**Choice of Datatset:**
Fashion-MNIST (70,000 grayscale images across 10 clothing categories).
- Fixed image sizes of 28x28 pixels.
- Up to 60,000 training images allows for a range in test cases when testing the breaking point of the claim
- Clothing categories are visually similar making classification more difficult


Run with over 5 randomly selected seeds. Mean and standard deviation calculated for a final representation of the result.

## Initial Experiment Set Up

In [1]:
%pip install torch
%pip install torchvision

#libray imports
import pathlib
import numpy as np
import matplotlib.pyplot as plt
import torch, torch.nn as nn
from torchvision import datasets, transforms

device = ('cuda' if torch.cuda.is_available() else
          'mps' if torch.backends.mps.is_available() else 'cpu')

transform = transforms.Compose([transforms.ToTensor()])
train_dataset = datasets.FashionMNIST('./data', train=True, download=True, transform=transform)
test_dataset = datasets.FashionMNIST('./data', train=False, download=True, transform=transform)
print(f'FashionMNIST: {len(train_dataset)} train images, {len(test_dataset)} test images')

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
FashionMNIST: 60000 train images, 10000 test images


### Nuisance Parameters

Controlled settings that are fixed for the experiment. 

Model Explanation:
256 hidden units gives:
- Input Layer: 784 neurons
- Hidden Layer 1: 256 neurons
- Hidden Layer 2: 256 neurons
- Output Layer: 10 neurons

This layout is complex enough to learn patterns in Fashion-MNIST.

In [2]:
#Settings
HIDDEN_UNITS = 256
TRAIN_PERCENT = 0.1
N_TRAIN = int(len(train_dataset)* TRAIN_PERCENT)
EPOCHS = 20 
BATCH = 128 #Mini-batch (balance between computation efficient and gradient stability)
LEARNING_RATE = 0.001 #Adam's standard
DROPOUT_RATES = [0.0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8]
SEED = [16, 31, 9, 92, 2]

## Tensor Loading

In [3]:
#Loading the dataset into device tensors: ... of the dataset will be used for training

def to_tensor(dataset, index=None):
    images, labels = (dataset.data, dataset.targets) if index is None else (dataset.data[index], dataset.targets[index])
    X = images.float().div(255).view(-1,784)
    return X.to(device), labels.to(device)

#generator = torch.Generator().manual_seed(16)
def random_sample(train_dataset, test_dataset):
    order = torch.randperm(len(train_dataset))[:N_TRAIN]
    x_train, y_train = to_tensor(train_dataset, order)
    x_test, y_test = to_tensor(test_dataset)
    print(f"device {device} | train {tuple(x_train.shape)} | test {tuple(x_test.shape)}")
    return x_train, x_test, y_train, y_test

## Training

In [4]:
#Making and training the model

#Why did we choose 2 hidden layers? we could change this ..., standardise this ... or explain why 2 is the best option.
def make_model(dropout_rate, seed):
    torch.manual_seed(seed)
    return torch.nn.Sequential(
        torch.nn.Linear(784, HIDDEN_UNITS), torch.nn.ReLU(), torch.nn.Dropout(dropout_rate),
        torch.nn.Linear(HIDDEN_UNITS, HIDDEN_UNITS), torch.nn.ReLU(), torch.nn.Dropout(dropout_rate),
        torch.nn.Linear(HIDDEN_UNITS, 10) #There are only 10 types of clothing class in fashionMNIST.
    ).to(device)

@torch.no_grad()
def accuracy(model, X, y):
    model.eval()
    return(model(X).argmax(1) == y).float().mean().item()

@torch.no_grad()
def calculate_loss(model, X, y):
    model.eval()
    loss_function = nn.CrossEntropyLoss()
    return loss_function(model(X), y).item()

def train(model, x_train, y_train, x_test, y_test, seed, verbose=True):
    random_generator = torch.Generator(device="cpu").manual_seed(seed)
    adam_opt = torch.optim.Adam(model.parameters(), lr = LEARNING_RATE)
    loss_function = nn.CrossEntropyLoss()
    results = {"train": [], 
               "test": [],
               "train_loss": [],
               "test_loss": []}
    for ep in range(EPOCHS):
        model.train()
        shuffle = torch.randperm(len(x_train), generator=random_generator).to(device)
        for i in range(0, len(x_train), BATCH):
            index = shuffle[i:i + BATCH]
            adam_opt.zero_grad()
            loss = loss_function(model(x_train[index]), y_train[index])
            loss.backward()
            adam_opt.step()
        results["train"].append(accuracy(model, x_train, y_train))
        results["test"].append(accuracy(model, x_test, y_test))
        results["train_loss"].append(calculate_loss(model, x_train, y_train))
        results["test_loss"].append(calculate_loss(model, x_test, y_test))
        if verbose and (ep + 1) % 10  ==0:
            print(
                f"epoch {ep + 1} | "
                f"Training Accuracy: {results['train'][-1]:.4f} | "
                f"Testing Accuracy: {results['test'][-1]:.4f} | "
                f"Training loss: {results['train_loss'][-1]:.4f} | "
                f"Testing Loss: {results['test_loss'][-1]:.4f}"
                
            )
    return results

## Running the Experiment

In [5]:
#Run with varying dropout
def dropout_experiment(seed):
    models = {}
    results = {}
    x_train, x_test, y_train, y_test = random_sample(train_dataset, test_dataset)

    for r in DROPOUT_RATES:
        models[r] = make_model(r, seed)
        results[r] = train(models[r], x_train, y_train, x_test, y_test, seed)

    print(f"{'Dropout Rate'} {'Training Accuracy'} {'Test Accuracy'} {'Difference'} {'Training Loss'} {'Testing Loss'}")
    for r in DROPOUT_RATES:
        train_data = results[r]["train"][-1]
        test_data = results[r]["test"][-1]
        train_loss = results[r]["train_loss"][-1]
        test_loss = results[r]["test_loss"][-1]
        print(f"{r:12.2f} {train_data:17.4f} {test_data:13.4f} {train_data-test_data:10.4f} {train_loss:12.4f} {test_loss:13.4f}")

    return results

In [6]:
def main():
    results = {}
    for i in range(0, len(SEED)):
        print(f"\nSeed: {SEED[i]}")
        results[SEED[i]] = dropout_experiment(SEED[i])
    return results

results = main()


Seed: 16
device mps | train (6000, 784) | test (10000, 784)
epoch 10 | Training Accuracy: 0.8997 | Testing Accuracy: 0.8364 | Training loss: 0.2789 | Testing Loss: 0.4714
epoch 20 | Training Accuracy: 0.9328 | Testing Accuracy: 0.8423 | Training loss: 0.1811 | Testing Loss: 0.4999
epoch 10 | Training Accuracy: 0.8888 | Testing Accuracy: 0.8344 | Training loss: 0.3018 | Testing Loss: 0.4867
epoch 20 | Training Accuracy: 0.9360 | Testing Accuracy: 0.8465 | Training loss: 0.1749 | Testing Loss: 0.4806
epoch 10 | Training Accuracy: 0.8793 | Testing Accuracy: 0.8270 | Training loss: 0.3182 | Testing Loss: 0.4966
epoch 20 | Training Accuracy: 0.9268 | Testing Accuracy: 0.8452 | Training loss: 0.1977 | Testing Loss: 0.4717
epoch 10 | Training Accuracy: 0.8718 | Testing Accuracy: 0.8257 | Training loss: 0.3351 | Testing Loss: 0.5049
epoch 20 | Training Accuracy: 0.9252 | Testing Accuracy: 0.8469 | Training loss: 0.2082 | Testing Loss: 0.4520
epoch 10 | Training Accuracy: 0.8697 | Testing Accu

## Final Result Comparision

In [7]:
#Code for mean and standard deviation across all results
results_final = {}
columns = ['Training Accuracy', 'Test Accuracy', 'Difference', 'Training Loss', 'Testing Loss']

def format(d):
    return {
        'Training Accuracy': d['train'][-1],
        'Test Accuracy' : d['test'][-1],
        'Difference' : d['train'][-1] - d['test'][-1],
        'Training Loss': d['train_loss'][-1],
        'Testing Loss': d['test_loss'][-1]
    }

for r in DROPOUT_RATES:
    results_final[r] = {}
    for i in range(0, 5):
        values = [format(results[s][r])[columns[i]] for s in SEED]
        results_final[r][columns[i]] = (np.mean(values), np.std(values, ddof=1))

print("Table displaying mean and standard deviation across all 5 seeds.")
print(f"{'Dropout Rate'} | {'Training Accuracy'} | {'Test Accuracy':17} | {'Difference':17} | {'Training Loss':17} | {'Testing Loss':17}")
for r in DROPOUT_RATES:
    print(f"{r:12.2f} | {results_final[r][columns[0]][0]:.4f}{results_final[r][columns[0]][1]:11.4f} | {results_final[r][columns[1]][0]:.4f}{results_final[r][columns[1]][1]:11.4f} | {results_final[r][columns[2]][0]:.4f}{results_final[r][columns[2]][1]:11.4f} | {results_final[r][columns[3]][0]:.4f}{results_final[r][columns[3]][1]:11.4f} | {results_final[r][columns[4]][0]:.4f}{results_final[r][columns[4]][1]:11.4f}")

Table displaying mean and standard deviation across all 5 seeds.
Dropout Rate | Training Accuracy | Test Accuracy     | Difference        | Training Loss     | Testing Loss     
        0.00 | 0.9276     0.0118 | 0.8400     0.0052 | 0.0876     0.0070 | 0.1993     0.0301 | 0.4975     0.0104
        0.10 | 0.9261     0.0103 | 0.8449     0.0039 | 0.0812     0.0087 | 0.1998     0.0267 | 0.4757     0.0140
        0.20 | 0.9196     0.0064 | 0.8457     0.0034 | 0.0739     0.0083 | 0.2179     0.0198 | 0.4660     0.0151
        0.30 | 0.9138     0.0118 | 0.8468     0.0058 | 0.0670     0.0114 | 0.2342     0.0276 | 0.4596     0.0218
        0.40 | 0.9047     0.0116 | 0.8453     0.0072 | 0.0594     0.0100 | 0.2563     0.0271 | 0.4585     0.0214
        0.50 | 0.8936     0.0089 | 0.8425     0.0054 | 0.0511     0.0092 | 0.2828     0.0262 | 0.4556     0.0146
        0.60 | 0.8848     0.0089 | 0.8403     0.0040 | 0.0444     0.0085 | 0.3123     0.0238 | 0.4531     0.0092
        0.70 | 0.8682     0.009

## Conclusion

For our network trained on 10% percent of data and no dropout we see that training accuracy and test accuracy were 93% and 84% respectively. Test loss was around 0.5 while training loss was at 0.2 and the generalisation gap 0.086. With the significantly higher test loss compared to lower training loss (0.199) we can see the network overfits. This is to be expected as the network has a much higher capacity than the 10% of the data needs.

With applying increasing dropout the most obvious trend is the improvement in the generalisation gap (0.088 for no dropout to 0.027 at 80% dropout). At each interval of dropout, the differnece between test accuracy and training accuracy falls, implying dropout reduces overfitting by making it harder for the model to memorise the training dataset. However, this reduction in gap occurs mainly due to the fall in training accuracy from 92% to 83%. The reason this occurs is because in each epoch a random subset of hidden units are turned off so the network does not rely on any particular units to fit individual training images, as descibed in 9.3.3.

Testing accuracy is highest at dropout of 30% because (85%) because this rate balances overfitting levels without removing too much capacity that might encourage underfitting. However, the best dropout level is not exactly defined as between 10-50% dropout they are within 0.5 of each other.

Training accuracy is lowest at 80% dropout (0.8332). This can be explained because with over 80% of neurons turned off there will only be approximately 50 working on each layer. This means that the network does not have enough capcity to fit the data and has noisy updates meaning underfitting has likely occured.

Test loss is 0.50 for no dropout and 0.45 for 50% dropout respectively. This means that the dropout networks make bettwr-quality predictions on unseen images so a lower test loss means the model is not as confident when wrong. It has a weaker effect for test accuracy and favours dropout levels of around 0.2 to 0.4, whereas test loss favours higher levels (0.5 to 0.6).

These results are comparable for the variation between the seeds as standard deviation never reaches above 0.2 meaning the accuracy can be conclusive. Overall, we can see that dropout rates from around 0.2 o 0.5 are the best at balancing fitting and generalisation.

Therefore, these results are the baseline of our experiment and confirm the theory in chapter 9.3.3. They show that overfitting occurs at 10% percent data (minimal training dataset) and that dropout reduces it. It sets the standard of improvment which we will compare against our experimental results as we increase the training dataset size.